# 00: Setup & Tokenization
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/gemma_from_scratch/blob/main/workshop/00_setup_and_tokenization.ipynb)

[Next: 01 The Math of Attention →](01_the_math_of_attention.ipynb)


**Estimated Time: 10 minutes**

A Language Model is a highly compressed, generative representation of human language and logic. To turn human language into math, the model must map words to numbers. This is where the tokenizer and the embedding lookup table act as the bridge between raw text and the generative engine. Before we can build the neural network, we must understand how text is converted into numerical indices that the model operates on. **Gemma 3 uses a 256K vocab SentencePiece BPE tokenizer** — the same base as the Gemini family.

> **Note:** **SentencePiece** (the framework) treats all text as a raw stream of Unicode characters. SentencePiece is completely language-agnostic, and it is lossless — meaning if you tokenize a sentence into integers and then decode it back, you get the exact original string, including all weird formatting, extra spaces, and line breaks. **BPE (Byte-Pair Encoding)** is, instead, a data compression technique adapted for machine learning. It builds the "lookup table" from the ground up through statistical frequency.

In our example we won't be using the Gemma tokenizer, but a simple replacement for didactic purposes.

--- 
## Learning Objectives
1. Understand how the tokenizer maps text to token IDs.
2. Visualize what tokens look like for different text.
3. Understand embedding size and the embedding-scaling trick.
4. Prepare the infrastructure for every subsequent notebook.

In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Set seed for reproducibility
_ = torch.manual_seed(42)

---
## 🧱 Deep Dive: Model Configuration & Parameter Estimation

In this cell, we define the core configuration parameters for our scaled-down pedagogical **~270M parameter Gemma 3** model. While commercial Gemma 3 models span 1B, 4B, and 27B parameters, we scale the dimensions down proportionally to run easily on local setups while preserving all major architectural patterns.

### 📐 Configuration Reference Table

| Parameter | Value | Architectural Purpose |
| :--- | :--- | :--- |
| `vocab_size` | `256,000` | Vocab width matching official Gemma 3 |
| `hidden_size` | `768` | Width of embedding vectors and hidden states ($d_{model}$) |
| `num_layers` | `8` | Number of Transformer blocks (alternating local/global) |
| `num_heads` | `8` | Query projection heads |
| `num_kv_heads` | `2` | Key/Value projection heads (Grouped Query Attention, 4:1 ratio) |
| `head_dim` | `96` | Dimensionality of each attention head ($768 / 8 = 96$) |
| `intermediate_size` | `2048` | Gated MLP (GeGLU) expansion factor |
| `sliding_window` | `1024` | Context window size for local SWA layers |
| `logit_cap` | `30.0` | Output logit capping factor using $30.0 \times \tanh(\text{logits}/30.0)$ |

### 🧮 Mathematical Breakdown of Parameter Counts

Transformers distribute parameters primarily across two sections: **embeddings** and **transformer blocks**:

1. **Token Embeddings ($W_{embed}$)**:

   $$\text{Embedding Parameters} = \text{vocab\_size} \times \text{hidden\_size} = 256,000 \times 768 = 196,608,000 \approx 196.6\text{M}$$

   > **Note:** Since we use weight tying, the LM head shares this exact parameter matrix, saving ~196.6M parameters.

2. **Per-Block Transformer Parameters**:
   Each layer contains attention and MLP sub-layers:
   - **GQA Projections**: 
     - Query: `hidden_size` $\times$ `hidden_size` $= 768 \times 768 = 589,824$
     - Key: `hidden_size` $\times$ (`num_kv_heads` $\times$ `head_dim`) $= 768 \times 192 = 147,456$
     - Value: `hidden_size` $\times$ (`num_kv_heads` $\times$ `head_dim`) $= 768 \times 192 = 147,456$
     - Output Projection: `hidden_size` $\times$ `hidden_size` $= 768 \times 768 = 589,824$
     - *Attention Total*: $1,474,560$ parameters.
   - **Gated MLP (GeGLU)**:
     - Gate Projection: `hidden_size` $\times$ `intermediate_size` $= 768 \times 2048 = 1,572,864$
     - Up Projection: `hidden_size` $\times$ `intermediate_size` $= 768 \times 2048 = 1,572,864$
     - Down Projection: `intermediate_size` $\times$ `hidden_size` $= 2048 \times 768 = 1,572,864$
     - *MLP Total*: $4,718,592$ parameters.
   - **RMSNorm Layers**: Four layers per block, each with a scale vector of size `hidden_size`:

     $$4 \times 768 = 3072\text{ parameters}$$

   - **Total per block**: $1,474,560 + 4,718,592 + 3072 = 6,196,224$ parameters.
3. **Total Model Weight Footprint (8 Layers)**:

   $$\text{Total Parameters} = \text{Embeddings} + (8 \times \text{Per-Block Total}) = 196.6\text{M} + (8 \times 6.2\text{M}) \approx 253.2\text{M}$$

   > **Note:** This is why the embedding vocabulary dominates the weight count of smaller models!

In [2]:
# Gemma 3 Architecture Configuration
# Scaled down proportionally from the 1B model for a ~270M target

vocab_size = 256_000  # Tokenizer vocabulary size
hidden_size = 768  # Embedding width (hidden dimension)
num_layers = 8  # Number of transformer blocks
num_attention_heads = 8  # Number of attention heads
num_kv_heads = 2  # Number of KV groups for GQA
intermediate_size = 2048  # FFN expansion factor
head_dim = hidden_size // num_attention_heads  # = 96

# Compute approx parameter count
emb_params = vocab_size * hidden_size

# Calculate Attention Params using proper GQA logic
q_params = hidden_size * hidden_size
k_params = hidden_size * (num_kv_heads * head_dim)
v_params = hidden_size * (num_kv_heads * head_dim)
o_params = hidden_size * hidden_size
attention_params = q_params + k_params + v_params + o_params

# Calculate MLP Params (GeGLU: Gate, Up, Down)
mlp_params = 3 * (hidden_size * intermediate_size)

# Calculate RMSNorm (4 per block)
rmsnorm_params = 4 * hidden_size

per_block_params = attention_params + mlp_params + rmsnorm_params

# Final Norm before the tied LM Head
final_norm = hidden_size

# Total Parameters (Note: LM Head is tied to embeddings, so it is omitted from the addition)
total_params = emb_params + (num_layers * per_block_params) + final_norm

print(f"Approximate parameter count: {total_params:,} ({total_params / 1e6:.1f}M)")
print(f"  - Embedding (tok_emb & lm_head tied): {emb_params / 1e6:.1f}M")
print(f"  - Per Block: {per_block_params / 1e6:.1f}M")

We calculate a total footprint of ~250M parameters when tracking both input and output projections. Embeddings are computed only once, because Gemma utilizes Weight Tying (sharing the exact same matrix memory state between the input tokenizer embeddings and the language model head):

Token Embeddings ($W_{embed}$):

$$\text{Embedding Parameters} = \text{vocab\_size} \times \text{hidden\_size} = 256,000 \times 768 = 196,608,000 \approx 196.6\text{M}$$

Per-Block Transformer Parameters (Totaling $\approx 6.2\text{M}$ parameters per layer):
1. GQA Projections: Query ($768^2$) + Key ($768 \times 192$) + Value ($768 \times 192$) + Output ($768^2$) = $1,474,560$ parameters.
2. Gated MLP (GeGLU): Gate ($768 \times 2048$) + Up ($768 \times 2048$) + Down ($2048 \times 768$) = $4,718,592$ parameters.
3. RMSNorm Layers: Four scale vectors per block = $4 \times 768 = 3,072$ parameters.

True Weight Footprint (with Tying):

$$\text{Total Trainable Footprint} = 196.6\text{M (Shared Matrix)} + (8 \times 6.2\text{M}) \approx 246.2\text{M}$$

---
## 1. The Tokenizer: Text → Indices

### 🔡 Understanding Tokenization & SimpleTokenizer Mechanics

A language model cannot directly process text characters. **Tokenization** is the process of breaking unstructured strings into structured subword units (tokens) and mapping each token to an integer ID from a pre-defined vocabulary list.

Gemma 3 uses a massive vocabulary of **256,000 tokens** to represent multi-lingual structures efficiently. In this cell, we implement a pedagogical `SimpleTokenizer` to model this process.

#### ⚙️ Mechanics of `SimpleTokenizer`:
1. **Vocabulary Mapping (`word2id` & `id2word`)**:
   - `word2id` maps each alphanumeric word string to a unique integer ID.
   - `id2word` provides the inverse mapping for decoding integer sequences back into readable words.
2. **Special Tokens (Gemma 3 Standard)**:
   - `<pad>` (`ID 0`): Used to pad unequal length batches to matching sequence lengths.
   - `<eos>` (`ID 1`): End of Sequence marker signaling that generation is complete.
   - `<bos>` (`ID 2`): Beginning of Sequence marker prepended to prime model context.
   - `<unk>` (`ID 3`): Unknown token placeholder for words outside our custom vocabulary.
3. **Encoding Pipeline**:
   - Text is split into words using space dividers.
   - Punctuation is stripped using standard alphanumeric filtering (`c.isalnum()`).
   - Clean words are converted to IDs. If a word is missing from our vocabulary list, it safely defaults to the `<unk>` ID (`3`).
4. **Decoding Pipeline**:
   - Maps integer arrays back into word strings.
   - Optionally filters out special tokens to present clean outputs to users.

In [3]:
_VOCAB = [
    # Special tokens (always first)
    "<pad>",
    "<eos>",
    "<bos>",
    "<unk>",
    # Common words
    "hello",
    "world",
    "what",
    "is",
    "are",
    "gemma",
    "model",
    "good",
    "work",
    "great",
    "better",
    "fast",
    "efficient",
    "amazing",
    "awesome",
    "build",
    "scratch",
    "train",
    "predict",
    "neural",
    "network",
    "attention",
    "layer",
    "hidden",
    "size",
    "embed",
    "token",
    "output",
    "sequence",
    "correct",
    "thanks",
    "this",
    "a",
    "all",
    "text",
    "make",
    "show",
    "learn",
    "forward",
    "pass",
    "embedding",
    "networks",
    "position",
    "decode",
    "generate",
    "input",
    "transform",
    "matrix",
    "compute",
    "loss",
    "optim",
    "grad",
    "step",
    "batch",
    "epoch",
    "architecture",
    "you",
    "need",
    "deep",
    "encoding",
    "masked",
    "causal",
    "normalization",
    "descent",
    "transformer",
    "models",
    "learning",
    "layers",
    "positional",
    "layers",
    "gradient",
    "optimizer",
]


class SimpleTokenizer:
    """Toy word-level tokenizer for the workshop"""

    # Fixed special token IDs — match Gemma 3 convention
    PAD_ID = 0
    EOS_ID = 1
    BOS_ID = 2
    UNK_ID = 3

    def __init__(self, vocab_size=vocab_size):
        self.word2id: dict[str, int] = {key: value for value, key in enumerate(_VOCAB)}
        self.id2word: dict[int, str] = {v: k for k, v in self.word2id.items()}
        self.vocab = self.word2id

    def encode(self, text: str, add_bos: bool = False) -> list[int]:
        """Tokenise a string into a list of integer token IDs"""
        ids = [self.BOS_ID] if add_bos else []
        for word in text.lower().split():
            clean = "".join(c for c in word if c.isalnum())
            ids.append(self.word2id.get(clean, self.UNK_ID))
        return ids

    def decode(self, ids: list[int], skip_special: bool = False) -> str:
        """Convert a list of token IDs back to a string"""
        special = {self.PAD_ID, self.EOS_ID, self.BOS_ID, self.UNK_ID}
        tokens = []
        for i in ids:
            if skip_special and i in special:
                continue
            tokens.append(self.id2word.get(i, "<unk>"))
        return " ".join(tokens)

    def __len__(self) -> int:
        """Number of tokens in the vocabulary."""
        return len(self.word2id)

    def __repr__(self) -> str:
        return f"SimpleTokenizer(vocab_size={len(self)})"

In [4]:
tokenizer = SimpleTokenizer()
print(f"Tokenizer vocabulary size: {len(tokenizer)}")
print(f"Special tokens: {tokenizer.vocab}")

### Testing the Tokenizer

In [5]:
text = "Hello world, what is Gemma model?"
token_ids = tokenizer.encode(text)
decoded = tokenizer.decode(token_ids)

print(f"Original text: {text}")
print(f"Token IDs:     {token_ids}")
print(f"Reconstructed: {decoded}")

### Token Distribution Visualization

In [6]:
corpus = [
    "neural networks are amazing",
    "transformer architecture",
    "gemma models are great",
    "attention is all you need",
    "deep learning is awesome",
    "token embedding layers",
    "positional encoding",
    "masked causal attention",
    "normalization layers",
    "gradient descent optimizer",
]

token_counts = {}
for sent in corpus:
    for tok in tokenizer.encode(sent):
        word = tokenizer.id2word.get(tok, "<unk>")
        token_counts[word] = token_counts.get(word, 0) + 1

words, counts = zip(*sorted(token_counts.items(), key=lambda x: x[1], reverse=True))
plt.figure(figsize=(12, 4))
plt.barh(range(len(words)), list(counts), align="center")
plt.yticks(range(len(words)), words)
plt.xlabel("Frequency")
plt.title("Token Frequencies in Corpus")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
## 2. The Embedding Layer

A model can't look up words directly — it looks up **vectors**. That's what `nn.Embedding` does: `token_id → vector of size hidden_size`.

> **Note:** Gemma 3 uses **weight tying**: the output head and the input embedding share the same weight matrix, saving ~2x vocab $	imes$ hidden_size parameters.

### 📐 Embedding Scaling (Gemma's Trick)

Gemma 3 multiplies the embedding by `sqrt(hidden_size)` at the very beginning. This is crucial:
- The embedding weights are initialized with small variance.
- The multiplication compensates, so the initial signal magnitude is reasonable.
- Without it, the first layer would receive tiny gradients and learn nothing.

Standard deep models initialize embedding distributions with very modest weight variances ($\sigma \approx 0.02$). Because of this, early attention dot-products skew exceptionally close to zero, producing weak signals that can stifle early-stage backpropagation.

Gemma 3 counters this by scaling token embeddings immediately upon generation:

$$\text{embedded\_scaled} = \text{embedded} \times \sqrt{d_{model}}$$

For this specific 768-dimensional implementation, it applies a multiplier of $\sqrt{768} \approx 27.7128$.

In [10]:
# Create the embedding layer
tok_emb = nn.Embedding(vocab_size, hidden_size)
torch.nn.init.normal_(tok_emb.weight, mean=0.0, std=1.0)

# Example: embed a sequence of tokens
sample_ids = torch.tensor([[tokenizer.encode("hello world go")]])
print(f"Input token IDs shape: {sample_ids.shape}")

embedded = tok_emb(sample_ids)
print(f"Embedded shape: {embedded.shape}")
# Shape: (batch, seq_len, hidden_size)
# For Gemma: each row is a 768-dim vector representing one token

In [14]:
print(
    f"Embeddings: mean={embedded.mean():.6f}, weight magnitude={embedded.abs().mean():.6f}, std={embedded.std():.6f}"
)

---
## 3. Training: Mini Training on Toy Data

Before building the full model, let's verify our embedding works with a simple training loop:
1. Take a short sequence.
2. Feed it through the embedding.
3. Compute a trivial loss (predict the next token).
4. Verify gradients flow.

In [9]:
import torch.nn.functional as F

# Simple next-token prediction loss
targets = torch.tensor([1])  # target: token ID 1 (the word "world")

# Compute a simple dot-product similarity and use cross-entropy
similarity = torch.matmul(torch.unsqueeze(embedded[0, 0, 0], 0), tok_emb.weight.T)

loss = F.cross_entropy(similarity, targets)

print(f"Initial loss: {loss.item():.4f}")

# Gradient check
loss.backward()
print(f"Gradients computed: {tok_emb.weight.grad is not None}")
print(
    f"Gradient stats: mean={tok_emb.weight.grad.mean():.6f}, std={tok_emb.weight.grad.std():.6f}"
)
print("\n✅ Embedding layer verified! Gradients flow correctly.")

This confirms that the `nn.Embedding` matrix parameters are fully tracking partial derivatives and are ready for downstream custom multi-head attention blocks!

---
## Summary

In this notebook, we:
1. **Configured Gemma 3 architecture** (~270M parameters) with all hyperparameters.
2. **Built a toy tokenizer** simulating the Gemma 3 SentencePiece BPE tokenizer (256K vocab).
3. **Implemented the embedding layer** with weight tying (shared tok_emb and out_head).
4. **Applied Gemma's embedding scaling** trick (`embedding` $\times$ `sqrt(hidden_size)`).
5. **Verified gradients flow** through the embedding layer.

In the next notebook, we'll build the core mechanism: **QK-Norm + Scaled Dot-Product Attention**.

[Next: 01 The Math of Attention →](01_the_math_of_attention.ipynb)